## 1. Data Overview

In [140]:
# 1.1 Load Data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../1_data/raw/Airline_review.csv")
print("Data loaded successfully!")

Data loaded successfully!


In [141]:
# 1.2 Shape & Data Types
print("=== Shape ===")
print(df.shape)

print("\n=== Data Types ===")
print(df.dtypes)

=== Shape ===
(23171, 20)

=== Data Types ===
Unnamed: 0                  int64
Airline Name               object
Overall_Rating             object
Review_Title               object
Review Date                object
Verified                     bool
Review                     object
Aircraft                   object
Type Of Traveller          object
Seat Type                  object
Route                      object
Date Flown                 object
Seat Comfort              float64
Cabin Staff Service       float64
Food & Beverages          float64
Ground Service            float64
Inflight Entertainment    float64
Wifi & Connectivity       float64
Value For Money           float64
Recommended                object
dtype: object


In [142]:
# 1.3 Data Structure
df.head()

,Unnamed: 0,Airline Name,Overall_Rating,Review_Title,Review Date,Verified,Review,Aircraft,Type Of Traveller,Seat Type,Route,Date Flown,Seat Comfort,Cabin Staff Service,Food & Beverages,Ground Service,Inflight Entertainment,Wifi & Connectivity,Value For Money,Recommended
0,0,AB Aviation,9,"""pretty decent airline""",11th November 2019,True,Moroni to Moheli. Turned out to be a pretty ...,NaN,Solo Leisure,Economy Class,Moroni to Moheli,November 2019,4.0,5.0,4.0,4.0,NaN,NaN,3.0,yes
1,1,AB Aviation,1,"""Not a good airline""",25th June 2019,True,Moroni to Anjouan. It is a very small airline...,E120,Solo Leisure,Economy Class,Moroni to Anjouan,June 2019,2.0,2.0,1.0,1.0,NaN,NaN,2.0,no
2,2,AB Aviation,1,"""flight was fortunately short""",25th June 2019,True,Anjouan to Dzaoudzi. A very small airline an...,Embraer E120,Solo Leisure,Economy Class,Anjouan to Dzaoudzi,June 2019,2.0,1.0,1.0,1.0,NaN,NaN,2.0,no
3,3,Adria Airways,1,"""I will never fly again with Adria""",28th September 2019,False,Please do a favor yourself and do not fly wi...,NaN,Solo Leisure,Economy Class,Frankfurt to Pristina,September 2019,1.0,1.0,NaN,1.0,NaN,NaN,1.0,no
4,4,Adria Airways,1,"""it ruined our last days of holidays""",24th September 2019,True,Do not book a flight with this airline! My fr...,NaN,Couple Leisure,Economy Class,Sofia to Amsterdam via Ljubljana,September 2019,1.0,1.0,1.0,1.0,1.0,1.0,1.0,no


In [143]:
# 1.4 Missing Values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_df.sort_values('missing_%', ascending=False)

,missing_count,missing_%
Wifi & Connectivity,17251,74.5
Aircraft,16042,69.2
Inflight Entertainment,12342,53.3
Food & Beverages,8671,37.4
Ground Service,4793,20.7
Cabin Staff Service,4260,18.4
Seat Comfort,4155,17.9
Route,3828,16.5
Date Flown,3754,16.2
Type Of Traveller,3738,16.1


> ### **(Note) Skytrax Review Data Characteristics**
> 
> The table below summarizes the input fields collected from the Skytrax airline review submission form and their characteristics. **This explains the reason that there are substantial missing values in certain columns.**
> 
> For reference, see the original review form: [Skytrax Review Form](https://www.airlinequality.com/write-a-review/?type=airline)

| Column | Scale | Notes |
|---|---|---|
| Airline Name | - | 497 unique airlines |
| Review_Title | - | Supplementary for analysis |
| Review Date | - | Scraping date, not flight date |
| Verified | True/False | Whether e-ticket or boarding pass was submitted |
| Review | 150~3500 chars | **Primary column for text analysis** |
| Aircraft | - | Optional field; high missingness |
| Type Of Traveller | - | Business / Family / Couple / Solo |
| Seat Type | - | First / Business / Premium Economy / Economy |
| Route | - | Free-text input |
| Date Flown | - | Actual flight date |
| Seat Comfort | 1~5 | Required field |
| Cabin Staff Service | 1~5 | Required field |
| Food & Beverages | 1~5 + N/A | N/A = service not available |
| Ground Service | 1~5 | Required field |
| Inflight Entertainment | 1~5 + N/A | N/A = service not available |
| Wifi & Connectivity | 1~5 + N/A | N/A = service not available |
| Value For Money | 1~5 | Required field |
| Overall_Rating | 1~10 | Different scale from sub-ratings |
| Recommended | Yes/No | **Target variable** |

## 2. Exploratory Data Analysis (EDA)

In [144]:
# 2.1 Target Variable (Recommended)
print(df['Recommended'].value_counts())
print(df['Recommended'].value_counts(normalize=True).round(3) * 100)

Recommended
no     15364
yes     7807
Name: count, dtype: int64
Recommended
no     66.3
yes    33.7
Name: proportion, dtype: float64


In [156]:
# 2.2 Known Categorical Columns

print("=== Verified ===")
print(df['Verified'].value_counts())

print("\n=== Seat Type ===")
print(df['Seat Type'].value_counts(dropna=False))

print("\n=== Type Of Traveller ===")
print(df['Type Of Traveller'].value_counts(dropna=False))

=== Verified ===
Verified
True     12322
False    10849
Name: count, dtype: int64

=== Seat Type ===
Seat Type
Economy Class      19145
Business Class      2098
NaN                 1096
Premium Economy      646
First Class          186
Name: count, dtype: int64

=== Type Of Traveller ===
Type Of Traveller
Solo Leisure      7120
Couple Leisure    5265
Family Leisure    4352
NaN               3738
Business          2696
Name: count, dtype: int64


In [146]:
# 2.3 Columns Requiring Investigation
# 2.3.1 Overall Rating
df["Overall_Rating"].value_counts(dropna=False)

Overall_Rating
1    11595
2     2296
9     1768
8     1757
3     1356
7     1193
4      859
n      842
5      830
6      675
Name: count, dtype: int64

> ### **Why Dropping `Overall_Rating` is needed?**
> 
> - **Missing 10**: Skytrax reviews use a 1 to 10 scale for `Overall_Rating`. The absence of 10-point ratings is unexplained and cannot be verified from the raw data alone.
> 
> - **Ambiguous 'n' values**: 842 entries contain 'n' which likely represents None or N/A, but cannot be reliably imputed or removed without introducing bias.
> 
> - **Data Leakage Risk**: `Overall_Rating` is logically correlated with the target variable `Recommended` (e.g., low ratings likely map to "no", high ratings to "yes"), which would inflate model performance and obscure the true predictive power of text-based sentiment features.

In [147]:
# 2.3.2 Review Date (by Year)
df['Review Date'] = pd.to_datetime(df['Review Date'], format='mixed', errors='coerce')
df['Review Date'].dt.year.value_counts().sort_index()

Review Date
2002      14
2003      45
2004     101
2005     136
2006     141
2007     168
2008     281
2009     287
2010     332
2011     461
2012     403
2013     494
2014     593
2015    1070
2016    1210
2017    1148
2018    1675
2019    2869
2020    1483
2021    1016
2022    3826
2023    5418
Name: count, dtype: int64

In [148]:
# 2.3.3 Date Flown (by Year)
df['Date Flown'] = pd.to_datetime(df['Date Flown'], format='%B %Y', errors='coerce')
df['Date Flown'].dt.year.value_counts().sort_index()

Date Flown
2012.0       1
2014.0      12
2015.0     891
2016.0    1155
2017.0    1160
2018.0    1774
2019.0    2929
2020.0    1378
2021.0    1022
2022.0    4115
2023.0    4980
Name: count, dtype: int64

In [149]:
# 2.3.4 Airline Name
print(f"Total unique airlines: {df['Airline Name'].nunique()}")
df['Airline Name'].value_counts().head(10)

Total unique airlines: 497


Airline Name
Caribbean Airlines     100
GoAir                  100
Germanwings            100
Philippine Airlines    100
Bangkok Airways        100
Garuda Indonesia       100
Batik Air              100
Swoop                  100
Frontier Airlines      100
Sunwing Airlines       100
Name: count, dtype: int64

In [150]:
# 2.3.5 Aircraft
print(f"Total unique aircraft: {df['Aircraft'].nunique()}")
df['Aircraft'].value_counts().head(10)

Total unique aircraft: 1048


Aircraft
A320              1041
Boeing 737-800     553
Boeing 737         404
A330               349
Boeing 787         349
A321               271
A319               233
Boeing 787-9       174
Boeing 777         160
A330-300           142
Name: count, dtype: int64

In [151]:
# 2.3.6 Route
print(f"Total unique routes: {df['Route'].nunique()}")
df['Route'].value_counts().head(10)

Total unique routes: 13607


Route
Melbourne to Sydney          43
Sydney to Melbourne          35
Cape Town to Johannesburg    34
Cusco to Lima                30
Bangkok to Phuket            28
Kuala Lumpur to Singapore    27
Johannesburg to Cape Town    27
Bangkok to Chiang Mai        26
Johannesburg to Durban       22
Toronto to Calgary           21
Name: count, dtype: int64

In [152]:
# 2.4 Numerical Rating Columns
rating_cols = ['Seat Comfort', 'Cabin Staff Service', 'Food & Beverages',
               'Ground Service', 'Inflight Entertainment',
               'Wifi & Connectivity', 'Value For Money']

df[rating_cols].describe()

,Seat Comfort,Cabin Staff Service,Food & Beverages,Ground Service,Inflight Entertainment,Wifi & Connectivity,Value For Money
count,19016.000000,18911.000000,14500.000000,18378.000000,10829.000000,5920.000000,22105.000000
mean,2.618374,2.871609,2.553586,2.353738,2.179056,1.780405,2.451165
std,1.464840,1.604631,1.526314,1.595747,1.488839,1.318800,1.594155
min,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
50%,3.000000,3.000000,2.000000,1.000000,2.000000,1.000000,2.000000
75%,4.000000,4.000000,4.000000,4.000000,3.000000,2.000000,4.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


> ### **Further Investigation Needed**
>
> **Reguired Columns (Scale 1 to 5)** → Both missing values and 0 are **NOT** accepted
> - Seat Comfort, Cabin Staff Service, Ground Service, Value For Money
>
> **Non-required Column (N/A or Scale 1 to 5)** → Missing values are accepted, but 0 is **NOT** accepted
> - Food & Beverages, Inflight Entertainment, Wifi & Connectivity

In [157]:
# 2.5 Further Investigation (Except for "Ground Service" as its min value is 1)
# 2.5.1 Seat Comfort
print("=== Seat Comfort ===")
print(df["Seat Comfort"].value_counts(dropna=False))

# 2.5.2 Cabin Staff Service
print("\n=== Cabin Staff Service ===")
print(df["Cabin Staff Service"].value_counts(dropna=False))

# 2.5.3 Value For Money
print("\n=== Value For Money ===")
print(df["Value For Money"].value_counts(dropna=False))

=== Seat Comfort ===
Seat Comfort
1.0    6337
NaN    4155
3.0    3618
4.0    3378
2.0    2869
5.0    2670
0.0     144
Name: count, dtype: int64

=== Cabin Staff Service ===
Cabin Staff Service
1.0    5971
5.0    4662
NaN    4260
4.0    2940
3.0    2848
2.0    2360
0.0     130
Name: count, dtype: int64

=== Value For Money ===
Value For Money
1.0    10136
5.0     3808
4.0     3379
2.0     2444
3.0     2201
NaN     1066
0.0      137
Name: count, dtype: int64


In [158]:
# 2.5.4 Food & Beverages
print("=== Food & Beverages ===")
print(df["Food & Beverages"].value_counts(dropna=False))

# 2.5.5 Inflight Entertainment
print("\n=== Inflight Entertainment ===")
print(df["Inflight Entertainment"].value_counts(dropna=False))

# 2.5.6 Wifi & Connectivity	
print("\n=== Wifi & Connectivity ===")
print(df["Wifi & Connectivity"].value_counts(dropna=False))

=== Food & Beverages ===
Food & Beverages
NaN    8671
1.0    5283
3.0    2424
4.0    2332
5.0    2242
2.0    1967
0.0     252
Name: count, dtype: int64

=== Inflight Entertainment ===
Inflight Entertainment
NaN    12342
1.0     4845
3.0     1700
4.0     1329
2.0     1278
5.0     1156
0.0      521
Name: count, dtype: int64

=== Wifi & Connectivity ===
Wifi & Connectivity
NaN    17251
1.0     4061
5.0      492
3.0      487
2.0      479
4.0      400
0.0        1
Name: count, dtype: int64
